## Bacillus
Nordholt N, van Heerden JH, Bruggeman FJ. Biphasic cell-size and growth-rate homeostasis by single Bacillus subtilis cells. Current Biology. 2020 Jun 22;30(12):2238-47.

In [1]:
# include packages
using LinearAlgebra
using DifferentialEquations, Plots
using BenchmarkTools
using HypothesisTests
default(grid=false,legend=false, size=(400,320), dpi=300, tickfontsize=9, lw=2, lc=:gray, labelfontsize=9, tickdir=:out, fontfamily="Arial", legendfontsize=7, markercolor=:indianred, thickness_scaling=2)

mydir = "../helper/"
include(mydir * "base.jl")
include(mydir * "optimization.jl")
include(mydir * "misc.jl")
include(mydir * "myplot.jl")


plot_output (generic function with 3 methods)

In [5]:
data = readdlm("../data/Bacillus/Glucose/Glucose_B15_0_IPTG.txt", ' ');
time = []
len = []
mother = []
id = []
time_temp = []
length_temp = []
mother_temp = []
id_temp = []
open("../data/Bacillus/Glucose/Glucose_B15_0_IPTG.txt", "r") do file
    for line in eachline(file)
        parts = split(line, '\t')
        push!(time_temp, parts[1])
        push!(length_temp, parts[3])    
        push!(mother_temp, parts[7])
        push!(id_temp, parts[6])
        if parts[2] == "1"
            push!(time, parse.(Float64, time_temp))
            push!(len, parse.(Float64, length_temp))
            push!(mother, mother_temp[1])
            push!(id, id_temp[1])
            
            empty!(time_temp)
            empty!(length_temp)
            empty!(mother_temp)
            empty!(id_temp)
        end
    end
end

In [6]:
mother = parse.(Float64, mother);
id = parse.(Float64, id);

In [7]:
traj = [];
for dau = 1:size(mother, 1)
    mom_place = findall(id.==mother[dau]);
    if !isempty(mom_place)
        push!(traj, [mom_place[1], dau])
    end
end

In [8]:
# iterate a few times
more_div = true;
while more_div
    more_div = false;
    for dau = 1:size(traj, 1)
        dau_place = findall(mother.==id[traj[dau][end]]);
        if !isempty(dau_place)
            push!(traj, copy(traj[dau]));
            push!(traj[dau], dau_place[1]);
            push!(traj[end], dau_place[2]);
            more_div = true;
        end
    end
end

In [9]:
traj_len = [];
traj_len_series = [];
traj_time = [];
traj_time_series = [];
for i = 1:size(traj,1)
    init_time = 0;
    for j in traj[i]
        push!(traj_len_series, len[j])
        push!(traj_time_series, time[j].+init_time)
        init_time = time[j][end] + init_time;
    end
    push!(traj_len, vcat(traj_len_series...))
    push!(traj_time, vcat(traj_time_series...))
    empty!(traj_len_series)
    empty!(traj_time_series)
end

traj_time = map(v -> convert(Vector{Float64}, v), traj_time);
traj_len = map(v -> convert(Vector{Float64}, v), traj_len);

traj_time = traj_time./60;

In [10]:
isdiv_, dtobs_, last_ = annotate_exp_traj2(traj_len, traj_time, cutoff = 0.5);

### trajectory

In [17]:
p_traj_obs = plot(; size=(750,300), xlim=(0,4), xtick=collect(0:2:4), ylim=(0,10), ytick=collect(0:5:10), bottommargin=-3Plots.mm,  leftmargin=-5Plots.mm, background_color=:transparent) # ,xlabel="Time", ylabel="Cell length"
for i in eachindex(traj_time)
    plot!(p_traj_obs, traj_time[i], traj_len[i], lc=:snow2, lw=0.2)
end
i = 10
plot!(p_traj_obs, traj_time[i], traj_len[i], lc=:gray, lw=.4)
scatter!(p_traj_obs, traj_time[i][isdiv_[i]], traj_len[i][isdiv_[i]], markercolor=:gray, ms=1, markerstrokewidth=0.2)
p_traj_obs
savefig("../figure/SI figure/bacillus/bacillus_traj.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/bacillus/bacillus_traj.pdf"

### correlation

In [19]:
using StatsBase
hedges = collect(2:0.5:8);
hbins = 0.5*( hedges[1:end-1] + hedges[2:end] )
dhedges = mean( diff(hedges) );
h2_obs = fit(Histogram, (vcat( [traj_len[i][isdiv_[i]][1:end-1] for i in eachindex(traj_len) ]...), vcat( [traj_len[i][isdiv_[i]][2:end] for i in eachindex(traj_len) ]...) ), (hedges,hedges));

h2_obs_normalize = h2_obs.weights / sum(h2_obs.weights) / dhedges^2

svd_h2_obs = svd(h2_obs_normalize);

ll=@layout [a b]
p_h2_obs = plot(; aspect_ratio=1, framestyle=:box, xlim=(2,8), ylim=(2,8), xtick=collect(2:3:8), ytick=collect(2:3:8))#, title="P(i,i-1)"
heatmap!( p_h2_obs, hbins, hbins, h2_obs_normalize, clim=(0,.5),c = :Blues_9,
colorbar_ticks=()) #,colorbar=true

plot(p_h2_obs, layout = ll, size=(300,300), dpi=300
, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
savefig("../figure/SI figure/bacillus/bacillus_correlation.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/bacillus/bacillus_correlation.png"

In [21]:
p_S_result = plot(; xlim=[0.5,10.5], ylim=[-0.1,1.05], ytick=[0,0.5,1], xtick=collect(1:4:13), 
size=(300,300), bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
scatter!( p_S_result, svd_h2_obs.S ./ svd_h2_obs.S[1], markercolor=:gray, ms=4, markershape=:hline,msw=1.5)

savefig("../figure/SI figure/bacillus/bacillus_svd.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/bacillus/bacillus_svd.pdf"

In [22]:
# mutual information
P_xy = h2_obs.weights/sum(h2_obs.weights)
P_y = sum(P_xy, dims=1)
P_x = sum(P_xy, dims=2)
is_nan = .!(isnan.(P_xy .* log.(P_xy./(P_x * P_y))))
mi = P_xy .* log.(P_xy./(P_x * P_y))
sum(mi[is_nan])

0.08863011681255295

### input data distribution

In [23]:
using StatsBase
hedges = collect(3:.2:7);
hbins = 0.5*( hedges[1:end-1] + hedges[2:end] )
dhedges = mean( diff(hedges) );
hh_obs = fit(Histogram, (vcat( [traj_len[i] for i in eachindex(traj_len) ]...), vcat( [last_[i] for i in eachindex(last_) ]...) ), (hedges,hedges));

hh_obs_normalize = hh_obs.weights / sum(hh_obs.weights) / dhedges^2

svd_hh_obs = svd(hh_obs_normalize);

ll=@layout [a b]
p_hh_obs = plot(; aspect_ratio=1, framestyle=:box, xlim=(3,7), ylim=(3,7), xtick=collect(3:2:7), ytick=collect(3:2:7))#, title="P(i,i-1)" 
heatmap!( p_hh_obs, hbins, hbins, hh_obs_normalize, clim=(0,.6),c = :Blues_9,
colorbar_ticks=(),colorbar=false) 

plot(p_hh_obs, layout = ll, size=(300,300), dpi=300
, bottommargin=-6Plots.mm, leftmargin=5Plots.mm, background_color=:transparent)
savefig("../figure/SI figure/bacillus/bcillus_datadist.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/bacillus/bcillus_datadist.png"

### Inference

In [144]:
bf1 = [sChebyshevT{Float64}(),sChebyshevT{Float64}()]
n_func1 = [4, 4]
l01 = [4.0, 4.0]
Θ1, Σ1 = mgs_gen_bases( [vcat(traj_len...), vcat(last_...)], n_func1, bf1, l01);
Θdiv1 = Θ1[vcat(isdiv_...),:];
Θdiv1 = sum(Θdiv1, dims=1)[:];

history1 = sparse_optimization_sweep(sparse_optimization_fg, Θ1, Θdiv1, vcat(dtobs_...); init_cutoff=0.0, coeffs0=1e-4, EPSILON=1E-7, MAX_ITERS=20);

===== start of epoch 1 =====


itr=1: 17466.632779938533


itr=2: 17432.310549853093


itr=3: 17434.381058151353


itr=4: 17436.814053509683


itr=5: 17437.406258739276


... (135 outputs left out)

itr=1: 19100.33159670484
itr=2: 19085.86703921435
itr=3: 19085.865505358463


itr=1: 23940.109503914562
itr=2: 23930.373132515935
itr=3: 23930.37291257561


itr=1: 23940.109503914562
itr=2: 23930.373132515935
itr=3: 23930.37291257561
===== end of epoch 10 =====
error = 0.059849370146612124, nterms = 1, cutoff = 1.204188855828028
===== start of epoch 11 =====


itr=1: 23940.109503914562
itr=2: 23930.373132515935
itr=3: 23930.37291257561
===== end of epoch 11 =====
error = 1.0, nterms = 0, cutoff = 4.525203477158637


In [65]:
save_inference_results([vcat(traj_time...) vcat(traj_len...) ], vcat(isdiv_...), vcat(dtobs_...), 
    [ [@sprintf "%s" typeof(bf0[i]) for i in 1:length(bf0)] n_func0 l00 ], history0; 
    data_type = "exp_bacillus", learn_type = "learn_memory0", output_dir = "./output/", 
    use_uid=true, save_traj=true
)

### plotting

In [24]:
filepath1 = "./output/exp_bacillus_learn_memory1/24-02-25-T2244/"

traj = readdlm(filepath1 * "traj");
tobs = traj[:,1];
Ls0 = [traj[:,j] for j in 2:2];
Ls1 = [traj[:,j] for j in 2:3];

dtobs = readdlm(filepath1 * "dts")[:];
isdiv = convert( BitVector, readdlm(filepath1 * "isdiv")[:] );

In [25]:
# rebuild inference basis function library

function rebuild_library_params(filepath)
    library_sum = readdlm(filepath * "library");
    bf_string = library_sum[:, 1];
    @show bf_string
    n_func = convert(Vector{Int}, library_sum[:,2]);
    l0 = convert(Vector{Float64}, library_sum[:,3]);
    return n_func, l0
end

n_func1, l01 = rebuild_library_params(filepath1);
bf1 = [sChebyshevT{Float64}(),sChebyshevT{Float64}()]
_, Σ1 = mgs_gen_bases(Ls1, n_func1, bf1, l01);

bf_string = Any["sChebyshevT{Float64}", "sChebyshevT{Float64}"]


In [26]:
# read in inference result
function rebuild_history(filepath)

    history_coeffs = readdlm(filepath * "history_coeffs");
    cache_summary  = readdlm(filepath * "history_summary");
    keynames = convert(Vector{String}, cache_summary[1,:] );
    history_metric = cache_summary[2:end,:];
    history = Dict(zip(Symbol.(keynames), [history_metric[:,i] for i = 1:length(keynames)]));

    return history, history_coeffs

end

history1, history_coeffs1 = rebuild_history(filepath1);


In [28]:
# find the maximum BIC score
_, imax1 = findmax( -history1[:error] .- 0.5*history1[:logdetH]/sum(dtobs) )

BIC = -history1[:error] .- 0.5*history1[:logdetH]/sum(dtobs)
plot(history1[:nterms], BIC, lw=2, lc=:dodgerblue4,
    xtick=[0,10,20],ytick=[-1,-0.5,0],ylim=(-1.1,0.1), size=(330,300), bottommargin=-4Plots.mm, leftmargin=-10Plots.mm,
    background_color=:transparent)
scatter!([history1[:nterms][imax1]], [BIC[imax1]], ms=2.5, mc=:dodgerblue4, shape=:rect)
savefig("../figure/SI figure/bacillus/bacillus_BIC.pdf")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/bacillus/bacillus_BIC.pdf"

In [29]:
# calculate the inferred λ
ll = collect(1.0:0.2:9.0);
ll_ = collect(1.0:0.2:9.0);
logλ_learn1 = mgs_rec_bases([ll, ll_], Σ1, history_coeffs1[:, imax1], bf1, l01);

pl1 = plot(;size=(300,300), bottommargin=-6Plots.mm, leftmargin=-3Plots.mm, background_color=:transparent);
heatmap!(pl1, ll, ll_, exp.(logλ_learn1'), clim=(0,3),
xlim=(3,7), ylim=(3,7), aspect_ratio=1, framestyle=:box, 
c=:lapaz, xtick=[3,5,7],ytick=[3,5,7])
savefig("../figure/SI figure/bacillus/bacillus_memory1.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/bacillus/bacillus_memory1.png"

In [30]:
heatmap(reshape(history_coeffs1[:, imax1],5,5), c =:broc, ticks=false, axis=true,aspect_ratio =1, clim=(-1,1),
xlim=(0.5,5.5), ylim=(0.5,5.5),framestyle=:box,size=(300,300), background_color=:transparent, yflip = true
)
savefig("../figure/SI figure/bacillus/bacillus_coeffs_matrix.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/bacillus/bacillus_coeffs_matrix.png"

### Growth function g inference

In [31]:
# interpolation to smooth the growth function g
function interpolation(t, L, deg)
    A = ones(length(L), deg);
    for i = 2:deg
        A[:,i] = t.^(i-1);
    end
    coeff = A\L;
    D = zeros(length(L), deg);
    for i = 2:deg
        D[:,i] = (i-1) * t.^(i-2);
    end
    return coeff, D * coeff, A * coeff
end

interpolation (generic function with 1 method)

In [32]:
bf_g = Polynomial()
Lt_g = []
Diff_g = []
for Li in eachindex(traj_len)
    pos = findall(isdiv_[Li]);
    for pi = 1:length(pos)-1
        _,diff, Lt = interpolation(traj_time[Li][pos[pi]+1:pos[pi+1]], traj_len[Li][pos[pi]+1:pos[pi+1]], 5);
        push!(Lt_g, Lt);
        push!(Diff_g, diff);
    end
end

# linear regression to learn growth rate function g
Lt_g = convert(Vector{Vector{Float64}}, Lt_g);
Diff_g = convert(Vector{Vector{Float64}}, Diff_g);
Lt_g = vcat( [ Lt_g[i]  for i in eachindex(Lt_g)]... );
Diff_g = vcat( [Diff_g[i] for i in eachindex(Diff_g) ]... );
Theta_g = gen_bases( Lt_g,  2);
coeffs_g = Theta_g[:,[1,2]]  \ Diff_g

2-element Vector{Float64}:
 -0.4567941701700083
  0.8367194351869558

In [38]:
histogram2d(Lt_g, Diff_g, framestyle=:box, normalize=:pdf, show_empty=true, size=(300,300), background_color=:transparent, bottommargin=-8Plots.mm, leftmargin=-2Plots.mm,
xlim = (1,7), ylims = (-2,8), xtick =[2, 4, 6], ytick=[0, 4, 8], bins=(0:.5:7, -5:.5:20), c=cgrad(:tempo, rev=false), clim = (0,.5)) # bins=(2:0.6:8, 0:3:30),aspect_ratio=0.2, c=:lapaz,
plot!([1,7], coeffs_g[1] .+ coeffs_g[2]*[1,7], lw=1.0, lc=:black, ls=:dash)
savefig("../figure/SI figure/bacillus/bacillus_infered_g.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/bacillus/bacillus_infered_g.png"

### Cut size function h inference

In [35]:
# linear regression to learn cut size function h
bf_h = Polynomial()
Y_h = vcat( [traj_len[i][isdiv_[i]] .- traj_len[i][findall(isdiv_[i]) .+  1]  for i in eachindex(traj_len)]... );
Theta_h = gen_bases( vcat( [traj_len[i][isdiv_[i]] for i in eachindex(traj_len) ]... ), 1);

coeffs_h = Theta_h\ Y_h
coeffs_h[abs.(coeffs_h) .< 0.02] .= 0.0

0-element view(::Vector{Float64}, Int64[]) with eltype Float64

In [39]:
histogram2d( vcat( [traj_len[i][isdiv_[i]] for i in eachindex(traj_len) ]... ), Y_h, framestyle=:box, normalize=:pdf, show_empty=true, size=(300,300), background_color=:transparent, bottommargin=-6Plots.mm, leftmargin=-2Plots.mm,
xlim = (3,7), ylims = (0,4.5), xtick =[3,5,7], ytick=[0, 2, 4], bins=(3:.25:7, 0:.3:6), c=cgrad(:tempo, rev=false),clim=(0,1)) # bins=(2:0.6:8, 0:3:30),aspect_ratio=0.2, c=:lapaz,)
plot!([3,7], coeffs_h[1] .+ coeffs_h[2] * [3,7].^1, lw=1.0, lc=:black, ls=:dash)
savefig("../figure/SI figure/bacillus/bacillus_infered_h.png")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/bacillus/bacillus_infered_h.png"

### resimulation

In [41]:
function f(du, u, p, t)
    bf_f = p[1]
    coeffs_f = p[2]
    du[1] = 0.0
    for n = 1:length(coeffs_f)
        du[1] += coeffs_f[n]*bf_f(n-1, u[1])
    end
end

function rate_memory(u, p, t)
    Σ1 = p[5]
    coeffs = p[6]
    bf1 = p[7]
    l01 = p[8]
    ln_rate = mgs_rec_bases([[u[1]], [u[2]]], Σ1, coeffs, bf1, l01);
    return exp(ln_rate[1])
end

function affect!(integrator)
    integrator.u[2] = integrator.u[1]
    bf_g = integrator.p[3]
    coeffs_g = integrator.p[4]
    u_old = integrator.u[1]
    for n = 1:length(coeffs_g)
        integrator.u[1] -= coeffs_g[n] * bf_g(n-1, u_old)
    end
end

affect! (generic function with 1 method)

In [85]:
# resimulate data and save or read in your resimulated data
# resimulate data based on your inferred parameters
Lresult = []
tresult = []
for i in eachindex(traj_len)
    vrj = VariableRateJump(rate_memory, affect!)
    oprob = ODEProblem(f, [traj_len[i][1];2 * traj_len[i][1]], (0.0,maximum(traj_time[i])), (bf_g, coeffs_g, bf_h, coeffs_h, Σ1, history_coeffs1[:, imax1], bf1, l01))
    jprob = JumpProblem(oprob, Direct(), vrj)
    sol = solve(jprob,Tsit5(),dtmax=0.01);
    L_r = Array(sol')[:,1];
    t_r = sol.t;
    push!(Lresult, L_r);
    push!(tresult, t_r);
end

Lresult = convert(Vector{Vector{Float64}}, Lresult);
tresult = convert(Vector{Vector{Float64}}, tresult);

isdiv_result = []
for i in eachindex(Lresult)
    isdiv_r, _ = annotate_sim_traj(Lresult[i], tresult[i])
    push!(isdiv_result, isdiv_r);
end

writedlm("../figure/SI figure/bacillus/bacillus_resim_data.txt", [Lresult, tresult], '\t')

In [86]:
# read in resimulated data

using JSON
resim_data = readdlm("../figure/SI figure/bacillus/bacillus_resim_data.txt", '\t')

Lresult = [Float64.(JSON.parse(resim_data[1, i])) for i in 1:265];
tresult = [Float64.(JSON.parse(resim_data[2, i])) for i in 1:265];

isdiv_result = []
for i in eachindex(Lresult)
    isdiv_r, _ = annotate_sim_traj(Lresult[i], tresult[i])
    push!(isdiv_result, isdiv_r);
end

In [87]:
p_traj = plot(; size=(750,300),xlim=(0,4), xtick=collect(0:2:4), ylim=(0,10), ytick=collect(0:5:10), bottommargin=-3Plots.mm,  leftmargin=-5Plots.mm, background_color=:transparent)
for i in eachindex(Lresult)
    plot!(p_traj, tresult[i], Lresult[i], lc=:snow2, lw=0.2)
end
plot!(p_traj, tresult[115], Lresult[115], lw = 0.4, lc=:seagreen)
scatter!(p_traj, tresult[115][isdiv_result[115]], Lresult[115][isdiv_result[115]], markercolor=:seagreen, ms=1, markerstrokewidth=0.2)
p_traj
savefig("../figure/SI figure/bacillus/bacillus_resim.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/bacillus/bacillus_resim.pdf"

In [90]:
using StatsBase
hedges = collect(2:0.5:8);
hbins = 0.5*( hedges[1:end-1] + hedges[2:end] )
dhedges = mean( diff(hedges) );
h1_result = fit(Histogram, vcat( [Lresult[i][isdiv_result[i]] for i in eachindex(Lresult) ]...), hedges)
h2_result = fit(Histogram, (vcat( [Lresult[i][isdiv_result[i]][1:end-1] for i in eachindex(Lresult) ]...), vcat( [Lresult[i][isdiv_result[i]][2:end] for i in eachindex(Lresult) ]...) ), (hedges,hedges));

h1_result_normalize = h1_result.weights / sum(h1_result.weights) / dhedges
h2_result_normalize = h2_result.weights / sum(h2_result.weights) / dhedges^2

svd_h1_result = svd(h1_result_normalize * h1_result_normalize');
svd_h2_result = svd(h2_result_normalize);

ll=@layout [a b]
p_h2_result = plot(; aspect_ratio=1, framestyle=:box, xlim=(2,8), ylim=(2,8), xtick=collect(2:3:8), ytick=collect(2:3:8) )#, title="P(i,i-1)"
heatmap!( p_h2_result, hbins, hbins, h2_result_normalize, clim=(0,.5),c = :Blues_9,
colorbar_ticks=()) #,colorbar=true

plot(p_h2_result, layout = ll, size=(300,300), dpi=300
, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
savefig("../figure/SI figure/bacillus/bacillus_correlation_resim2.png")


"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/bacillus/bacillus_correlation_resim2.png"

In [91]:
p_S_result = plot(; xlim=[0.5,10.5], ylim=[-0.1,1.05], ytick=[0,0.5,1], xtick=collect(1:4:13), 
size=(300,300), bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)
scatter!( p_S_result, svd_h2_obs.S ./ svd_h2_obs.S[1], markercolor=:gray, ms=4, markershape=:hline,msw=1.5)
scatter!( p_S_result, svd_h2_result.S ./ svd_h2_result.S[1], markercolor=:seagreen, ms=3.5, markershape=:circle,msw=1, msc=:seagreen)
savefig("../figure/SI figure/bacillus/bacillus_svd_resim.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/bacillus/bacillus_svd_resim.pdf"

In [92]:
# mutual information
P_xy = h2_result.weights/sum(h2_result.weights)
P_y = sum(P_xy, dims=1)
P_x = sum(P_xy, dims=2)
is_nan = .!(isnan.(P_xy .* log.(P_xy./(P_x * P_y))))
mi = P_xy .* log.(P_xy./(P_x * P_y))
sum(mi[is_nan])

0.15321143663445944

### histogram

In [93]:
p1 = plot(;xlim=(2,9),  xtick=collect(2:3:10),ylim=(-0.02,0.8),ytick=[0,0.4,0.8])
stephist!( p1, vcat( [traj_len[i][isdiv_[i]] for i in eachindex(traj_len) ]...), bins=collect(2:0.5:9), normalize=:pdf, lw=0.2#, xlabel="Cell length", ylabel="PDF"
, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)

stephist!( p1, vcat( [Lresult[i][isdiv_result[i]] for i in eachindex(Lresult) ]...), normalize=:pdf, bins=collect(1:0.5:9), lc=:seagreen, lw=0.2)
savefig("../figure/SI figure/bacillus/bacillus_histogram_divsize.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/bacillus/bacillus_histogram_divsize.pdf"

In [94]:
p2 = plot()
stephist!( p2, vcat( diff.([traj_time[i][isdiv_[i]] for i in eachindex(traj_time)]) ... ), normalize=:pdf,
        xlim=(0,1.82), xtick=collect(0:0.8:1.8),ylim=(-0.02,2.5),ytick=[0,1,2], bins=collect(0:0.16:1.8), lw=0.2
        , bottommargin=-6Plots.mm, leftmargin=0Plots.mm, background_color=:transparent)
        #,ylabel="PDF", xlabel="Generation time" )
stephist!( p2, vcat( diff.([tresult[i][isdiv_result[i]] for i in eachindex(tresult)]) ... ), normalize=:pdf,
xlim=(0,1.82), xtick=collect(0:0.8:1.8),ylim=(-0.02,2.5),ytick=[0,1,2], bins=collect(0:0.16:1.8), lw=0.2
, lc=:seagreen)

savefig("../figure/SI figure/bacillus/bacillus_histogram_gentime.pdf")

"/Users/shijiezhang/Desktop/cell_growth_code_final/figure/SI figure/bacillus/bacillus_histogram_gentime.pdf"

### histogram

In [25]:
p1 = plot(;xlim=(2,9),  xtick=collect(2:3:10),ylim=(-0.02,0.8),ytick=[0,0.4,0.8])
stephist!( p1, vcat( [traj_len[i][isdiv_[i]] for i in eachindex(traj_len) ]...), bins=collect(2:0.5:9), normalize=:pdf, lw=0.2#, xlabel="Cell length", ylabel="PDF"
, bottommargin=-6Plots.mm, leftmargin=-6Plots.mm, background_color=:transparent)

stephist!( p1, vcat( [Lresult_[i][isdiv_result_[i]] for i in eachindex(Lresult_) ]...), normalize=:pdf, bins=collect(1:0.5:9), lc=:seagreen, lw=0.2)
savefig("bacillus_histogram_divsize.pdf")

"/Users/shijiezhang/Desktop/NJSDE/cell_growth/figure/SI figure/bacillus_histogram_divsize.pdf"

In [35]:
p2 = plot()
stephist!( p2, vcat( diff.([traj_time[i][isdiv_[i]] for i in eachindex(traj_time)]) ... ), normalize=:pdf,
        xlim=(0,1.82), xtick=collect(0:0.2:1.8),ylim=(-0.02,2.5),ytick=[0,1,2], bins=collect(0:0.1:1.8), lw=0.2
        , bottommargin=-6Plots.mm, leftmargin=0Plots.mm, background_color=:transparent)
        #,ylabel="PDF", xlabel="Generation time" )
stephist!( p2, vcat( diff.([tresult_[i][isdiv_result_[i]] for i in eachindex(tresult_)]) ... ), normalize=:pdf
, lc=:seagreen, bins=collect(0:0.1:1.6), lw=0.2)

savefig("bacillus_histogram_gentime.pdf")

"/Users/shijiezhang/Desktop/NJSDE/cell_growth/figure/SI figure/bacillus_histogram_gentime.pdf"

In [41]:
display(plot(Lresult_[1]))